In [ ]:
#load packages
import os
import numpy as np
import pandas as pd


In [ ]:
#file paths

CLIMATE_DIR = "/group/moniergrp/dbaral"

file_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "LOCA_csv")
save_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data","LOCA_gdd")

#get all the files in filet path
all_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith(".csv")]

In [ ]:
# base temp, lower temp, optimum temp and threshold gdd for each rice stage: needed to calculated daily gdd
#temp are in C
#referenced from Sharifi et al 2016 - for medium grain rice
# Pl_PI: Planting(PL) to Panicle Initiation(PI)
#PI_HD: Panicle Initiation (PI) to Heading (HD)
#HD_R7: Heading(HD) to R7 stage

growth_stages = {
    'PL_PI' : {
       'tbase': 10,
       'tlower': 14.30,
       'topt': 27.73,
       'threshold': 454
   },
   'PI_HD' : {
       'tbase': 9.8,
       'tlower': 15.37,
       'topt': 30.73,
       'threshold': 356
   },
   'HD_R7' : {
       'tbase': 8.9,
       'tlower': 15.93,
       'topt': 33.03,
       'threshold': 203
   }
}

In [ ]:
#function to calculate daily gdd
def calculate_daily_gdd (tmmn, tmmx, tbase, tlower, topt):
  tmmx_adj = min(tmmx, topt)
  tmmn_adj = min(tmmn, tlower)
  tmean_adj = (tmmx_adj + tmmn_adj)/2
  gdd = max(tmean_adj - tbase, 0)
  return gdd

In [ ]:
#process each county and year combination

for file in all_files:  
    df = pd.read_csv(file)

    file_results = []
    
    for (county, year), group in df.groupby(['county', 'year']):
        season_data = group.copy()
        season_data = season_data.sort_values(by= 'date')

        #Convert 'date' column to datetime objects for reliable comparison
        season_data['date'] = pd.to_datetime(season_data['date'])

        #setting planting date to May 15 and harvest date to sep 15
        year_int = int(year)
        planting_date = pd.to_datetime(f"{year_int}-05-15")
        harvest_date  = pd.to_datetime(f"{year_int}-09-15")

        # Now you can filter within the season
        season_data = season_data[
            (season_data['date'] >= planting_date) &
            (season_data['date'] <= harvest_date)
        ]

        #Initialize growth tracking
        current_stage = 'PL_PI'
        stage_gdd = 0
        accumulated_gdd = 0
        transition_dates = {
            'pl_date': season_data.iloc[0]['date'] if not season_data.empty else None, # To handle empty season_data
            'pi_date': None,
            'hd_date': None,
            'r7_date': None,
            'grain_fill_start_date': None,
            'grain_fill_end_date': None
        }

        # Handling case where season_data might be empty after filtering
        if season_data.empty:
            print(f"No data for growing season in {county}, {year}")
            continue # Skip to the next group if no data for the season

        #daily process loop

        for index, row in season_data.iterrows():
            # Calculate daily GDD based on the current stage's parameters
            stage_params = growth_stages.get(current_stage)
            daily_gdd = 0

            if stage_params:
                daily_gdd = calculate_daily_gdd(
                    row['tmmn'],
                    row['tmmx'],
                    stage_params['tbase'],
                    stage_params['tlower'],
                    stage_params['topt']
                )
                stage_gdd += daily_gdd
                accumulated_gdd += daily_gdd


            # Check for stage transition based on GDD
            new_stage = None
            if current_stage in growth_stages:
                stage_params = growth_stages.get(current_stage)
                if stage_params and stage_gdd >= stage_params['threshold']:
                    if current_stage == 'PL_PI':
                        transition_dates['pi_date'] = row['date']
                        new_stage = 'PI_HD'
                    elif current_stage == 'PI_HD':
                        transition_dates['hd_date'] = row['date']
                        new_stage = 'HD_R7'
                    elif current_stage == 'HD_R7':
                        transition_dates['r7_date'] = row['date']
                        new_stage = 'completed' # Transition to a completed stage

            # Update current stage and reset stage_gdd if a transition occurred
            if new_stage:
                current_stage = new_stage
                stage_gdd = 0 # Reset stage_gdd for the new stage

            # Check for grain fill start date based on hd_date + 0 days
            if current_stage == 'completed' and transition_dates['hd_date'] is not None and transition_dates['grain_fill_start_date'] is None:
                transition_dates['grain_fill_start_date'] = transition_dates['hd_date']
                transition_dates['grain_fill_end_date'] = transition_dates['grain_fill_start_date'] + pd.Timedelta(days=30)


            # If in the grain fill period, update the stage
            if transition_dates['grain_fill_start_date'] is not None and transition_dates['grain_fill_end_date'] is not None:
                if row['date'] >= transition_dates['grain_fill_start_date'] and row['date'] <= transition_dates['grain_fill_end_date']:
                    current_stage = 'grain_fill'

            # save record
            if row['date'] <= harvest_date:
                # Add to overall results
                file_results.append({
                    'county': county,
                    'date': row['date'],
                    'year': year,
                    'month': row['month'],
                    'day': row['day'],
                    'stage': current_stage, # Use current_stage after transition check
                    'daily_gdd': daily_gdd,
                    'accumulated_gdd': accumulated_gdd,
                    'stage_gdd': stage_gdd, # stage_gdd is reset upon transition
                    'pl_date': transition_dates['pl_date'],
                    'pi_date': transition_dates['pi_date'],
                    'hd_date': transition_dates['hd_date'],
                    'r7_date': transition_dates['r7_date'],
                    'grain_fill_start_date': transition_dates['grain_fill_start_date'],
                    'grain_fill_end_date': transition_dates['grain_fill_end_date']
                })
    results_df = pd.DataFrame(file_results)
    
    #output_name = os.path.basename(file).replace(".csv","")+"_gdd_results.csv"
    #output_path = os.path.join(save_path, output_name)
    #results_df.to_csv(output_path, index = False)
    
    date_cols = ['pl_date', 'pi_date', 'hd_date', 'r7_date', 'grain_fill_start_date', 'grain_fill_end_date']
    results_df[date_cols] = results_df[date_cols].apply(pd.to_datetime)
    

    #calculating day of the year for each of the date cols
    for col in date_cols:
        results_df[f'{col}_doy'] = results_df[col].dt.dayofyear


    #converting date column to datetime format
    results_df['date'] = pd.to_datetime(results_df['date'])
    

    # Calculate days after planting for pi_date and hd_date
    results_df['pi_dap'] = (results_df['pi_date'] - results_df['pl_date']).dt.days
    results_df['hd_dap'] = (results_df['hd_date'] - results_df['pl_date']).dt.days
    results_df['r7_dap'] = (results_df['r7_date'] - results_df['pl_date']).dt.days
    results_df['grain_fill_dap'] = (results_df['grain_fill_start_date'] - results_df['pl_date']).dt.days

    results_df.to_csv(file_path + '/Rice_Growth_Stages_GDD_Results_LOCA2.csv', index=False)
    
    output_name = os.path.basename(file).replace("rice_temp_county_daily_means.csv","")+"_rice_growth_stages_gdd_results_LOCA2.csv"
    output_path = os.path.join(save_path, output_name)
    results_df.to_csv(output_path, index = False)
    print(f"Saved {output_name} {len(results_df)} rows") 